# 모델 1. 단순 선형회귀 (녹지 비율만)
**작성자: A**

질문: **녹지 하나로 얼마나 설명되나?**

6주차 3단계 '베이스라인 먼저': 서울 평균(아무것도 안 함) 기준선도 여기서 같이 돌림

In [ ]:
# ── 공통 설정 (모든 노트북 맨 위에 그대로 두기) ──
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AICEE_project'   # 모두 같은 경로 (공유 폴더를 '내 드라이브에 바로가기 추가' 해야 함)
DATA = f'{BASE}/data_processed'
RESULTS = f'{BASE}/results'

import sys
sys.path.append(f'{BASE}/shared')
RANDOM_STATE = 42

USE_DUMMY = True   # ← 진짜 데이터 나오면 False로 바꾸기

> **강의 규칙 (5·6주차)** — 튜닝은 교차검증 점수로만 · 전처리는 Pipeline 안에 · 바꾼 설정은 `note`에 기록 · 테스트 데이터는 C만

In [ ]:
FEATURES = ['green_300', 'impervious_300', 'water_300', 'dist_han_m']   # 확정되면 팀 공통으로 수정
TARGETS = ['uhi_day', 'uhi_night']   # 낮·밤 열섬 강도

In [ ]:
import pandas as pd
df = pd.read_csv(f'{DATA}/train_dummy.csv' if USE_DUMMY else f'{DATA}/train.csv')
print(df.shape); df.head()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from evaluate import cv_by_district, save_cv_result, worst_sensors

X_COLS = ['green_300']
NOTE = ''   # ← 설정 바꿀 때마다 적기 (6주차: 시도한 모든 설정 기록)
results = {}
for target in TARGETS:
    model = make_pipeline(StandardScaler(), LinearRegression())
    overall, per_gu, oof = cv_by_district(model, df[X_COLS], df[target], groups=df['gu'])
    save_cv_result('model1_LR', target, overall, BASE, note=NOTE)
    results[target] = (overall, per_gu, oof)
    print(target, {k: round(v, 3) for k, v in overall.items()})

## 기준선: 서울 평균

In [ ]:
for t in TARGETS:
    o, _, _ = cv_by_district(DummyRegressor(strategy='mean'), df[X_COLS], df[t], groups=df['gu'])
    save_cv_result('baseline_mean', t, o, BASE)
    print(t, {k: round(v, 3) for k, v in o.items()})

## 잔차 분석 (3주차: 잔차 그림이 정직한 검사)

In [ ]:
import matplotlib.pyplot as plt
overall, per_gu, oof = results['uhi_night']
resid = df['uhi_night'] - oof
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(df['uhi_night'], oof, s=8, alpha=0.5); ax[0].plot([-2, 3], [-2, 3], 'k--', lw=1)
ax[0].set_xlabel('true UHI (night, °C)'); ax[0].set_ylabel('predicted'); ax[0].set_title('predicted vs true')
ax[1].scatter(oof, resid, s=8, alpha=0.5); ax[1].axhline(0, color='k', lw=1)
ax[1].set_xlabel('predicted'); ax[1].set_ylabel('residual (true - pred)'); ax[1].set_title('residuals (3주차: 무늬가 보이면 모델이 놓친 것)')
plt.tight_layout()
# plt.savefig(f'{RESULTS}/X_fig1_residuals.png', dpi=200)
plt.show()
per_gu.sort_values('RMSE', ascending=False).head()   # 어느 구에서 특히 틀리나 (부분집단 분석)

## 실패 사례 후보 (6주차 7단계: 최소 3개)
오차가 큰 센서를 로드뷰·지도로 확인해서 **왜 틀렸는지** 메모

In [ ]:
worst_sensors(df, df['uhi_night'], results['uhi_night'][2], k=10)

## 보고서용 메모 (결과 나오자마자 2~3문장씩)
- 결과 (RMSE 평균 ± 표준편차):
- 한계:
- 실패 사례:
- 생성형 AI 사용 내역 (도구, 프롬프트, 수정한 부분):